# 1. Train, Validation, & Test Splits

### Roles of Each Dataset Split:
- **Training Set (60-80%):** Used strictly to fit model weights/parameters.
- **Validation Set (10-20%):** Used to tune hyperparameters and evaluate model generalization during development.
- **Test Set (10-20%):** Unseen vault evaluation set used *once* at the end to estimate real-world deployment performance.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("Cleaned_Validated_Data.csv")
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Sub-split Train into Train/Validation (75/25 of Train -> 60/20/20 overall)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

print(f"Total Rows: {len(df)}")
print(f"Train Set: {len(X_tr)} | Validation Set: {len(X_val)} | Test Set: {len(X_test)}")

Total Rows: 1010
Train Set: 606 | Validation Set: 202 | Test Set: 202


# 2. Stratified Sampling vs. Temporal Splitting

### Concepts:
- **Stratified Sampling:** Ensures class proportion distributions in target $y$ are preserved identically across Train, Validation, and Test splits.
- **Temporal Splitting:** Splitting time-series data chronologically (e.g., Train on past months, Test on future months) to avoid predicting the past using future data.

In [2]:
# Stratified Split
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Original Target Proportions:\n", y.value_counts(normalize=True).round(4))
print("Stratified Test Proportions:\n", y_test_strat.value_counts(normalize=True).round(4))

Original Target Proportions:
 Churn
No     0.696
Yes    0.304
Name: proportion, dtype: float64
Stratified Test Proportions:
 Churn
No     0.698
Yes    0.302
Name: proportion, dtype: float64


# 3. Correct Preprocessing Fit/Transform Rules

> **Golden Rule:** ALWAYS `fit()` or `fit_transform()` transformers **strictly on the Training Set**. Apply `transform()` on Validation and Test sets using the parameters learned from training.

In [3]:
from sklearn.preprocessing import StandardScaler

# Correct Workflow Execution
X_num_tr = X_tr[["TenureYears", "MonthlyCharges"]].fillna(0)
X_num_val = X_val[["TenureYears", "MonthlyCharges"]].fillna(0)
X_num_ts = X_test[["TenureYears", "MonthlyCharges"]].fillna(0)

scaler = StandardScaler()

# FIT strictly on Training Set
X_num_tr_scaled = scaler.fit_transform(X_num_tr)

# TRANSFORM Validation and Test Sets using training mean & std
X_num_val_scaled = scaler.transform(X_num_val)
X_num_ts_scaled = scaler.transform(X_num_ts)

print("Scaler Mean learned strictly from Training Set:", scaler.mean_.round(2))
print("Notebook 12 execution completed successfully!")

Scaler Mean learned strictly from Training Set: [ 4.21 57.38]
Notebook 12 execution completed successfully!
